# 🃏 Previsão de Preço de Cartas Pokémon TCG
Pipeline de ML para prever `tcgplayer.prices.holofoil.market`.

## Parte 1 — Coleta dos Dados
Usando a [Pokémon TCG API](https://pokemontcg.io/) v2.

In [ ]:
import requests
import pandas as pd
import time
import json
from pathlib import Path

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

API_BASE = 'https://api.pokemontcg.io/v2/cards'
TIMEOUT = 30
MAX_RETRIES = 3

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')
API_KEY = os.getenv('POKEMON_TCG_API_KEY', '')

headers = {
    'Content-Type': 'application/json',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
}
if API_KEY:
    headers['X-Api-Key'] = API_KEY

resp = requests.get(API_BASE, params={'page': 1, 'pageSize': 1}, headers=headers, timeout=TIMEOUT)
print(f'Status: {resp.status_code}')
total = resp.json().get('totalCount', 0)
print(f'Cartas disponíveis: {total:,}')

In [ ]:
def fetch_page(page, page_size=250):
    """Tenta buscar uma página com retry."""
    params = {'page': page, 'pageSize': page_size, 'q': 'supertype:Pokemon'}
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(API_BASE, headers=headers, params=params, timeout=TIMEOUT)
            if resp.status_code == 200:
                return resp.json().get('data', [])
            print(f'  Tentativa {attempt}/{MAX_RETRIES} — status {resp.status_code}')
        except Exception as e:
            print(f'  Tentativa {attempt}/{MAX_RETRIES} — erro: {e}')
        time.sleep(2 ** attempt)  # backoff: 2s, 4s, 8s
    return None

def fetch_all_cards(max_pages=50, page_size=250):
    """Baixa N páginas e retorna lista de cartas."""
    all_cards = []
    for page in range(1, max_pages + 1):
        data = fetch_page(page, page_size)
        if data is None:
            print(f'  Página {page}: excedeu retries, parando.')
            break
        if not data:
            print(f'  Página {page}: vazia, parando.')
            break
        all_cards.extend(data)
        print(f'  Página {page}: {len(data)} cartas (total: {len(all_cards)})')
        time.sleep(0.5)
    return all_cards

cards = fetch_all_cards(max_pages=10, page_size=250)
print(f'\nTotal coletado: {len(cards)} cartas')

In [ ]:
with open(DATA_DIR / 'cards_raw.json', 'w') as f:
    json.dump(cards, f)
print(f'Salvo em {DATA_DIR / "cards_raw.json"}')

In [ ]:
def extract_features(card):
    prices = card.get('tcgplayer', {}).get('prices', {})
    holofoil = prices.get('holofoil', {})
    normal = prices.get('normal', {})
    return {
        'id': card['id'],
        'name': card.get('name'),
        'supertype': card.get('supertype'),
        'subtypes': ', '.join(card.get('subtypes', [])),
        'hp': card.get('hp'),
        'types': ', '.join(card.get('types', [])),
        'rarity': card.get('rarity'),
        'artist': card.get('artist'),
        'set_id': card.get('set', {}).get('id'),
        'set_name': card.get('set', {}).get('name'),
        'set_series': card.get('set', {}).get('series'),
        'set_release_date': card.get('set', {}).get('releaseDate'),
        'set_printed_total': card.get('set', {}).get('printedTotal'),
        'number': card.get('number'),
        'national_pokedex': card.get('nationalPokedexNumbers', []),
        'legalities_standard': card.get('legalities', {}).get('standard'),
        'legalities_expanded': card.get('legalities', {}).get('expanded'),
        'regulation_mark': card.get('regulationMark'),
        'price_holofoil_market': holofoil.get('market'),
        'price_holofoil_mid': holofoil.get('mid'),
        'price_normal_market': normal.get('market'),
        'price_normal_mid': normal.get('mid'),
    }

df = pd.DataFrame([extract_features(c) for c in cards])
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
has_price = df['price_holofoil_market'].notna()
print(f'Cartas com preço holofoil: {has_price.sum()} / {len(df)}')
print(f'Cartas com preço normal:   {df["price_normal_market"].notna().sum()}')
if has_price.any():
    print(df['price_holofoil_market'].describe().to_frame('preço holofoil'))